In [28]:
from transformers import BertTokenizer, BertForMaskedLM
import torch
import torch.nn.functional as F



In [29]:
tokenizor=BertTokenizer.from_pretrained("bert-base-uncased")
model=BertForMaskedLM.from_pretrained("bert-base-uncased")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [30]:
text="The capital city of france is [MASK]"

In [31]:
input=tokenizor(text,return_tensors="pt")

In [32]:
with torch.inference_mode():
    logits=model(**input).logits

In [33]:
mask_idx=(input.input_ids == tokenizor.mask_token_id).nonzero(as_tuple=True)[1]
mask_index = mask_idx[0].item() 

In [34]:
mask_logits = logits[0, mask_index, :]

In [35]:
probs=F.softmax(mask_logits,dim=-1)
top_k=torch.topk(probs,k=5,dim=-1)

In [36]:
print("Top 5 predictions for [Mask]")
for token_id, prob in zip(top_k.indices,top_k.values):
    token=tokenizor.decode([token_id.item()])
    print(f"{token:<12}->{prob.item():.4f}")

Top 5 predictions for [Mask]
.           ->0.7253
;           ->0.2424
|           ->0.0284
?           ->0.0017
!           ->0.0017
